# 04 Error analysis

Where does the chosen tree model go wrong? Everything is computed on the test set of each split from the fitted
candidates saved by `train`, and every sentence in the summary is generated from those numbers.

In [1]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from house_prices import clean, config, load

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", context="notebook")
config.PATHS.figures_dir.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    fig.savefig(config.PATHS.figures_dir / f"{name}.png", dpi=140, bbox_inches="tight")

In [2]:
import json

import joblib
from IPython.display import display

from house_prices import evaluate, features, formatting, split

metrics = json.loads(config.PATHS.metrics_json.read_text())
frames = {}
for setup in split.SETUPS:
    train_df, test_df = split.load_split(setup)
    chosen = metrics["setups"][setup]["chosen_model"]
    model = joblib.load(config.PATHS.candidates_dir / f"{setup}_{chosen}.joblib")
    frames[setup] = evaluate.build_error_frame(test_df, train_df, model.predict(features.make_inputs(test_df)))
    print(setup, evaluate.MODEL_LABELS[chosen], f"{len(test_df):,} test listings")

random XGBoost 13,415 test listings


time LightGBM 13,549 test listings


## Predicted against actual, and residuals

In [3]:
for setup, frame in frames.items():
    label = evaluate.MODEL_LABELS[metrics["setups"][setup]["chosen_model"]]
    fig = evaluate.plot_predicted_vs_actual(frame, setup, label)
    plt.show()
    fig = evaluate.plot_residuals(frame, setup, label)
    plt.show()

## Error by city, property type, price decile and location coverage

In [4]:
for setup, frame in frames.items():
    analysis = metrics["setups"][setup]["error_analysis"]
    label = evaluate.MODEL_LABELS[metrics["setups"][setup]["chosen_model"]]
    fig = evaluate.plot_error_by_group(analysis, setup, label)
    plt.show()
    for key in ("by_city", "by_property_type", "by_price_decile", "by_location_listings"):
        table = pd.DataFrame(analysis[key])
        table["mdape"] = (table["mdape"] * 100).round(1)
        table["bias_pct"] = (table["bias_pct"] * 100).round(1)
        table["mae_pkr"] = table["mae_pkr"].round(0)
        print(f"{setup} split: {key}")
        display(table)

random split: by_city


,group,n,mdape,mape,mae_pkr,bias_pct
0,Faisalabad,374,16.8,0.244858,3077157.0,1.9
1,Islamabad,2069,12.5,0.191663,6029800.0,-1.2
2,Karachi,5008,14.4,0.215732,4919139.0,-1.1
3,Lahore,4624,12.7,0.173962,4601848.0,0.6
4,Rawalpindi,1340,14.6,0.202085,2930808.0,-0.1


random split: by_property_type


,group,n,mdape,mape,mae_pkr,bias_pct
0,Farm House,19,53.4,0.761701,37635314.0,11.1
1,Flat,3370,15.7,0.225534,2317416.0,-0.8
2,House,9536,13.0,0.187207,5590539.0,-0.2
3,Lower Portion,132,10.3,0.156687,1935152.0,-0.1
4,Penthouse,51,20.7,0.231480,5983182.0,-15.7
5,Upper Portion,307,11.3,0.167727,3488815.0,-0.9


random split: by_price_decile


...

,group,n,mdape,mape,mae_pkr,bias_pct
0,Farm House,24,42.2,0.685816,30313333.0,28.9
1,Flat,3379,17.4,0.295305,2391636.0,2.5
2,House,9621,13.9,0.208202,5521527.0,0.3
3,Lower Portion,152,15.4,0.234149,2488979.0,-2.2
4,Penthouse,45,26.9,0.439530,8818598.0,12.9
5,Upper Portion,328,14.3,0.217771,2502158.0,2.1


time split: by_price_decile


,group,n,mdape,mape,mae_pkr,bias_pct
0,1,1357,24.8,0.524982,1145427.0,22.6
1,2,1369,16.9,0.248450,1197994.0,6.2
2,3,1375,14.4,0.221573,1502829.0,1.5
3,4,1430,15.3,0.228811,2052212.0,2.9
4,5,1375,13.4,0.190168,2258173.0,-1.3
5,6,1346,13.4,0.183644,2726169.0,-2.0
6,7,1254,12.0,0.167640,3189957.0,-0.8
7,8,1367,12.6,0.173412,4536646.0,-4.5
8,9,1321,12.6,0.171883,6627751.0,-1.2
9,10,1355,16.0,0.204296,21845329.0,-6.2


time split: by_location_listings


,group,n,mdape,mape,mae_pkr,bias_pct
0,not in training data,76,25.8,0.325396,5035121.0,3.6
1,1-19 listings (grouped as Other),1508,17.7,0.260844,3686575.0,-1.0
2,20-49 listings,981,14.3,0.244810,3176298.0,0.9
3,50-199 listings,2689,13.8,0.232993,4513588.0,0.6
4,200+ listings,8295,14.3,0.224165,5103355.0,1.0


## Bias by price decile

Positive values mean the model over-predicts that band on average; negative means it under-predicts. Trees pull
extreme prices toward the middle, so the cheapest deciles tend to be over-predicted and the most expensive
under-predicted. The table shows whether that happens here.

In [5]:
for setup, frame in frames.items():
    decile = frame.groupby("price_decile").agg(listings=("ape", "size"), median_error=("ape", "median"), bias=("log_ratio", lambda v: np.exp(v.median()) - 1))
    fig, ax = plt.subplots(figsize=(7, 3.4))
    sns.barplot(x=decile.index, y=decile["bias"] * 100, ax=ax, color="#4c78a8")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xlabel("Price decile (1 = cheapest)")
    ax.set_ylabel("Median bias (%)")
    ax.set_title(f"Median over/under-prediction by price decile ({setup} split)")
    plt.show()

## The ten worst predictions

In [6]:
for setup in split.SETUPS:
    worst = pd.DataFrame(metrics["setups"][setup]["error_analysis"]["worst_10"])
    show = worst.assign(actual=[formatting.format_pkr(v) for v in worst["actual_price"]], predicted=[formatting.format_pkr(v) for v in worst["predicted_price"]], error=(worst["ape"] * 100).round(0).astype(int).astype(str) + "%")
    print(f"{setup} split")
    display(show[["city", "location", "property_type", "area_sqft", "bedrooms", "baths", "actual", "predicted", "error", "location_train_listings", "likely_reason"]])

random split


,city,location,property_type,area_sqft,bedrooms,baths,actual,predicted,error,location_train_listings,likely_reason
0,Islamabad,F-11,House,1361.250,3,3,PKR 10 Lakh,PKR 1.3 Crore,1199%,399,Asking price per sq ft is 13.7x below the usua...
1,Karachi,Bahria Town Karachi,Flat,1143.450,2,2,PKR 6.5 Lakh,PKR 65.45 Lakh,907%,1357,Asking price per sq ft is 9.6x below the usual...
2,Karachi,Sea View Apartments,House,25047.000,3,3,PKR 3.5 Crore,PKR 27.69 Crore,691%,51,Asking price per sq ft is 9.8x below the usual...
3,Karachi,Gulistan E Jauhar,Flat,1932.975,3,3,PKR 15 Lakh,PKR 1.12 Crore,648%,2288,Asking price per sq ft is 7.7x below the usual...
4,Islamabad,G-15,Flat,762.300,1,1,PKR 7.35 Lakh,PKR 48.24 Lakh,556%,267,Asking price per sq ft is 7.4x below the usual...
5,Karachi,Dastgir Colony,House,1306.800,4,3,PKR 20 Lakh,PKR 1.31 Crore,554%,25,Asking price per sq ft is 4.3x below the usual...
6,Karachi,Bahria Town Karachi,Flat,1143.450,2,2,PKR 10 Lakh,PKR 63.38 Lakh,534%,1357,Asking price per sq ft is 6.2x below the usual...
7,Karachi,Bahria Town Karachi,Flat,1987.425,3,3,PKR 20.5 Lakh,PKR 1.22 Crore,497%,1357,Asking price per sq ft is 5.3x below the usual...
8,Karachi,North Nazimabad,Flat,1687.950,2,2,PKR 18 Lakh,PKR 1.01 Crore,460%,1207,Asking price per sq ft is 6.7x below the usual...
9,Karachi,Gulshan E Iqbal Town,Upper Portion,60984.000,4,4,PKR 3.25 Crore,PKR 17.4 Crore,435%,1937,Asking price per sq ft is 14.0x below the usua...


time split


,city,location,property_type,area_sqft,bedrooms,baths,actual,predicted,error,location_train_listings,likely_reason
0,Islamabad,G-6,Flat,1824.075,3,3,PKR 9 Lakh,PKR 2.31 Crore,2464%,33,Asking price per sq ft is 24.8x below the usua...
1,Islamabad,F-7,House,10890.000,7,8,PKR 1 Crore,PKR 21.88 Crore,2088%,135,Asking price per sq ft is 20.0x below the usua...
2,Karachi,Bahria Town Karachi,House,3811.500,4,0,PKR 15 Lakh,PKR 1.66 Crore,1004%,1318,Asking price per sq ft is 14.2x below the usua...
3,Karachi,Bahria Town Karachi,Flat,2368.575,3,3,PKR 13 Lakh,PKR 1.34 Crore,933%,1318,Asking price per sq ft is 10.2x below the usua...
4,Islamabad,Gulberg,Flat,598.950,1,1,PKR 3.75 Lakh,PKR 38.5 Lakh,927%,190,Asking price per sq ft is 10.1x below the usua...
5,Islamabad,Gulberg,Flat,925.650,1,1,PKR 5.5 Lakh,PKR 54.34 Lakh,888%,190,Asking price per sq ft is 10.7x below the usua...
6,Karachi,Bahria Town Karachi,Flat,1143.450,2,2,PKR 6.5 Lakh,PKR 63.71 Lakh,880%,1318,Asking price per sq ft is 9.8x below the usual...
7,Islamabad,B-17,House,1361.250,4,4,PKR 9.5 Lakh,PKR 85.81 Lakh,803%,206,Asking price per sq ft is 7.4x below the usual...
8,Karachi,Bahria Town Karachi,Flat,3566.475,4,4,PKR 16.7 Lakh,PKR 1.49 Crore,791%,1318,Asking price per sq ft is 12.0x below the usua...
9,Karachi,Bahria Town Karachi,Flat,3566.475,4,4,PKR 16.8 Lakh,PKR 1.49 Crore,786%,1318,Asking price per sq ft is 11.9x below the usua...


The reasons are automatic guesses from simple rules (price per sq ft against the location's usual level,
area per bedroom, and location coverage). They point at likely data entry problems; they are not verified.

## Summary of the weakest areas

In [7]:
for setup in split.SETUPS:
    print(f"{setup} split")
    for line in metrics["setups"][setup]["error_analysis"]["weakest"]:
        print("-", line)
    print()

random split
- By city, the weakest group is 'Faisalabad' with a median error of 16.8% (n=374) against 13.6% overall; the strongest is 'Islamabad' at 12.5%.
- By property type, the weakest group is 'Penthouse' with a median error of 20.7% (n=51) against 13.6% overall; the strongest is 'Lower Portion' at 10.3%.
- By price decile, the weakest group is '1' with a median error of 18.3% (n=1384) against 13.6% overall; the strongest is '7' at 10.4%.
- By location coverage, the weakest group is 'not in training data' with a median error of 21.4% (n=63) against 13.6% overall; the strongest is '200+ listings' at 13.1%.

time split
- By city, the weakest group is 'Karachi' with a median error of 16.8% (n=5188) against 14.6% overall; the strongest is 'Lahore' at 13.0%.
- By property type, the weakest group is 'Penthouse' with a median error of 26.9% (n=45) against 14.6% overall; the strongest is 'House' at 13.9%.
- By price decile, the weakest group is '1' with a median error of 24.8% (n=1357) ag

## What the model relies on (SHAP)

In [8]:
from PIL import Image

for name in ("shap_bar", "shap_beeswarm", "shap_waterfall_example"):
    path = config.PATHS.figures_dir / f"{name}.png"
    if path.exists():
        plt.figure(figsize=(9, 5.5))
        plt.imshow(Image.open(path))
        plt.axis("off")
        plt.show()
shap_block = metrics.get("shap")
if shap_block:
    print(shap_block["example_sentence"])
    display(pd.DataFrame(shap_block["group_importance"]).round(4))

Property type pulled this price down the most (-25%), while location pushed it up (+30%).


,group,mean_abs_shap
0,Size,0.5392
1,Location,0.2291
2,Property type,0.1241
3,Bedrooms,0.0850
4,Bathrooms,0.0613
5,Distance to city centre,0.0590
6,City,0.0219
